# Flujo de trabajo integrado con `Pipeline` y `ColumnTransformer`

Las funciones `Pipeline` y `make_pipeline` permiten encadenar procesos de manera **secuencial**.

Se usan principalmente para empaquetar el preprocesamiento con el modelado de los datos, lo que evita que haya [fuga de datos](https://www.ibm.com/mx-es/think/topics/data-leakage-machine-learning), particularmente en la etapa de sintozación de hiperparámetros por validación cruzada.

Vamos a ver cómo usar estas funciones junto con `ColumnTransformer`.

In [1]:
import pandas as pd

df = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data-original",
    sep=r'\s+',
    header=None,
    names=['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 
    'model_year', 'origin', 'car_name']
    )

df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name
0,18.0,8.0,307.0,130.0,3504.0,12.0,70.0,1.0,chevrolet chevelle malibu
1,15.0,8.0,350.0,165.0,3693.0,11.5,70.0,1.0,buick skylark 320
2,18.0,8.0,318.0,150.0,3436.0,11.0,70.0,1.0,plymouth satellite
3,16.0,8.0,304.0,150.0,3433.0,12.0,70.0,1.0,amc rebel sst
4,17.0,8.0,302.0,140.0,3449.0,10.5,70.0,1.0,ford torino


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 406 entries, 0 to 405
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     406 non-null    float64
 2   displacement  406 non-null    float64
 3   horsepower    400 non-null    float64
 4   weight        406 non-null    float64
 5   acceleration  406 non-null    float64
 6   model_year    406 non-null    float64
 7   origin        406 non-null    float64
 8   car_name      406 non-null    object 
dtypes: float64(8), object(1)
memory usage: 28.7+ KB


Con este dataset vamos a entrenar un modelo que prediga la variable **mpg**.

Como esta variable tiene datos nulos, vamos a eliminar algunos registros.

In [3]:
df.dropna(subset=['mpg'], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 398 entries, 0 to 405
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    float64
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        398 non-null    float64
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    float64
 7   origin        398 non-null    float64
 8   car_name      398 non-null    object 
dtypes: float64(8), object(1)
memory usage: 31.1+ KB


### Paso 1: Crear $X$, $y$, y partir los datos

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop([
    'mpg',
    'car_name',
    'horsepower'
    ], axis=1)

y = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1, train_size=0.7)  # noqa: E501
print(f'Tamaño del conjunto de entrenamiento es: {X_train.shape}')
print(f'Tamaño del conjunto de prueba es: {X_test.shape}')

Tamaño del conjunto de entrenamiento es: (278, 6)
Tamaño del conjunto de prueba es: (120, 6)


### Paso 2: Instanciar los transformadores que se van a usar

Vamos a hacer las siguientes transformciones:

- Las variables 'displacement', 'weight' y 'acceleration' se van a estandarizar con `StandardScaler`.
- La variable 'origin' se va a codificar con `OneHotEncoder`.
- Las variables 'cylinders' y 'model_year' se van a codificar con `OrdinalEncoder`.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

ohe = OneHotEncoder(sparse_output=False, drop='first')
oe_cyl = OrdinalEncoder(categories=[[3., 4., 5., 6., 8.]])
oe_my = OrdinalEncoder(categories=[[70., 71., 72., 73., 74., 75., 76., 77., 78., 79., 80., 81., 82.]])  # noqa: E501
ss = StandardScaler()

preprocessor = ColumnTransformer(transformers=[
    ('ohe', ohe, ['origin']), # Codificacióp OneHot para la variable 'origin
    ('oe_cylinders', oe_cyl, ['cylinders']), # Codificación ordinal para la variable 'cylinders'  # noqa: E501
    ('oe_model_year', oe_my, ['model_year']), # Codificación ordinal para la variable 'model_year'  # noqa: E501
    ('ss', ss, ['displacement', 'weight', 'acceleration'])
    ], # Escalamiento de las variables numéricas
    remainder='passthrough') # El resto de las columnas se mantienen sin cambios

preprocessor

,transformers,"[('ohe', ...), ('oe_cylinders', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,False


### Paso 3: Instanciar modelo y empaquetar usando `Pipeline`

In [6]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

clf = Ridge()

model = Pipeline([
    ('preprocessor', preprocessor),
     ('clf', clf)
    ])

model

,steps,"[('preprocessor', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('ohe', ...), ('oe_cylinders', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Paso 4: Sintonizar hiperparámetros y entrenar modelo

In [7]:
import numpy as np
from sklearn.model_selection import GridSearchCV

alpha = np.logspace(-4, 2)

grid = {
    'clf__alpha':alpha, # Noten que hay que indicar a que paso corresponde el hiperparámetro  # noqa: E501
    }

search = GridSearchCV(
    estimator=model, # Se define el modelo a sintonizar, que incluye el preprocesador y el modelo Ridge  # noqa: E501
    param_grid=grid, # Se define la grilla de hiperparámetros que se va a sintonizar
    scoring='neg_root_mean_squared_error', # Se define la métrica de evaluación, en este caso RMSE  # noqa: E501
    refit=True, # Se entrenará modelo con los mejores hiperparámetros
    )

search.fit(X_train, y_train)

print(f'Mejor valor de RMSE de validación cruzada: {-search.best_score_:.3f}')
print(f'Mejores hiperparámetros: {search.best_params_}')

Mejor valor de RMSE de validación cruzada: 3.461
Mejores hiperparámetros: {'clf__alpha': np.float64(0.0001)}


### Paso 5: Evaluar el mejor modelo con los datos de entrenamiento y prueba

In [8]:
from sklearn.metrics import root_mean_squared_error

best_model = search.best_estimator_
print(f'Error de entrenamiento: {root_mean_squared_error(y_train, best_model.predict(X_train)):.3f}')  # noqa: E501
print(f'Error de prueba: {root_mean_squared_error(y_test, best_model.predict(X_test)):.3f}')  # noqa: E501

Error de entrenamiento: 3.368
Error de prueba: 2.999


Los coeficientes asociados a cada característica del modelo son:

In [9]:
pesos = pd.DataFrame(
    data=search.best_estimator_.named_steps['clf'].coef_, # Se extraen los coeficientes del modelo Ridge  # noqa: E501
    index=search.best_estimator_.named_steps['preprocessor'].get_feature_names_out().tolist(), # Se extraen los nombres de las variables  # noqa: E501
    columns=['Coeficiente']
    )

pesos.loc['intercepto'] = search.best_estimator_.named_steps['clf'].intercept_ # Se extrae el intercepto del modelo Ridge  # noqa: E501
pesos

,Coeficiente
ohe__origin_2.0,2.800455
ohe__origin_3.0,2.243036
oe_cylinders__cylinders,-1.214958
oe_model_year__model_year,0.846356
ss__displacement,3.663709
ss__weight,-6.658319
ss__acceleration,0.520269
intercepto,20.371044


## Flujos de trabajo más complejos

Se pueden crear flujos más complejos usando estratégicamente `Pipeline` y `ColumnTransformer`. Por ejemplo supongamos que vamos a añadir la variable 'horsepower' a nuestra matriz de características y queremos hacer el siguiente
procesamiento:


- Las variables 'displacement', 'weight', 'horsepower' y 'acceleration' se van a estandarizar con `StandardScaler`.
- Se va a imputar la media de 'horsepower' a los valores nulos de esta variable.
- Se van a crear características cuadráticas a partir de las variables 'displacement', 'weight' y 'horsepower'.
- La variable 'origin' se va a codificar con `OneHotEncoder`.
- Las variables 'cylinders' y 'model_year' se van a codificar con `OrdinalEncoder`.

Como añadimos una nueva variable, es necesario crear nuevamente $X$, $y$ y partir los datos:

In [ ]:
X = df.drop([
    'mpg',
    'car_name',
    ], axis=1)

y = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1, train_size=0.7)  # noqa: E501
print(f'Tamaño del conjunto de entrenamiento es: {X_train.shape}')
print(f'Tamaño del conjunto de prueba es: {X_test.shape}')

Tamaño del conjunto de entrenamiento es: (278, 7)
Tamaño del conjunto de prueba es: (120, 7)


Ahora instanciamos los transformadores:

In [11]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PolynomialFeatures

ohe = OneHotEncoder(sparse_output=False, drop='first')
oe_cyl = OrdinalEncoder(categories=[[3., 4., 5., 6., 8.]])
oe_my = OrdinalEncoder(categories=[[70., 71., 72., 73., 74., 75., 76., 77., 78., 79., 80., 81., 82.]])  # noqa: E501
ss = StandardScaler()
imputer = SimpleImputer(strategy='mean')
poly = PolynomialFeatures(degree=2, include_bias=False)

proc_num = Pipeline([
    ('imputer', imputer),
    ('poly', poly),
    ('ss', ss)
    ])

preprocessor = ColumnTransformer(transformers=[
    ('ohe', ohe, ['origin']), # Codificacióp OneHot para la variable 'origin
    ('oe_cylinders', oe_cyl, ['cylinders']), # Codificación ordinal para la variable 'cylinders'  # noqa: E501
    ('oe_model_year', oe_my, ['model_year']), # Codificación ordinal para la variable 'model_year'  # noqa: E501
    ('proc_num', proc_num, ['displacement', 'weight', 'horsepower', 'acceleration'])
    ], # Escalamiento de las variables numéricas
    remainder='drop') # El resto de las columnas se mantienen sin cambios

preprocessor

,transformers,"[('ohe', ...), ('oe_cylinders', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,False


Volvemos a empaquetar todo:

In [12]:
clf = Ridge()

model = Pipeline([
    ('preprocessor', preprocessor),
     ('clf', clf)
    ])

model

,steps,"[('preprocessor', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('ohe', ...), ('oe_cylinders', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


Sintonizamos hiperparámetros y entrenamos:

In [13]:
alpha = np.logspace(-4, 2)

grid = {
    'clf__alpha':alpha,
    }

search = GridSearchCV(
    estimator=model,
    param_grid=grid,
    scoring='neg_root_mean_squared_error',
    refit=True,
    )

search.fit(X_train, y_train)

print(f'Mejor valor de RMSE de validación cruzada: {-search.best_score_:.3f}')
print(f'Mejores hiperparámetros: {search.best_params_}')

Mejor valor de RMSE de validación cruzada: 3.077
Mejores hiperparámetros: {'clf__alpha': np.float64(0.20235896477251575)}


Evaluamos el modelo:

In [14]:
best_model = search.best_estimator_
print(f'Error de entrenamiento: {root_mean_squared_error(y_train, best_model.predict(X_train)):.3f}')  # noqa: E501
print(f'Error de prueba: {root_mean_squared_error(y_test, best_model.predict(X_test)):.3f}')  # noqa: E501


Error de entrenamiento: 2.848
Error de prueba: 2.862


Obtenemos los coeficientes del modelo:

In [15]:
pesos = pd.DataFrame(
    data=search.best_estimator_.named_steps['clf'].coef_,
    index=search.best_estimator_.named_steps['preprocessor'].get_feature_names_out().tolist(),
    columns=['Coeficiente']
    )

pesos.loc['intercepto'] = search.best_estimator_.named_steps['clf'].intercept_
pesos

,Coeficiente
ohe__origin_2.0,1.478746
ohe__origin_3.0,0.829590
oe_cylinders__cylinders,0.311646
oe_model_year__model_year,0.864392
proc_num__displacement,-1.216573
proc_num__weight,-8.323718
proc_num__horsepower,-6.999789
proc_num__acceleration,-2.972621
proc_num__displacement^2,-1.787317
proc_num__displacement weight,4.873294
